# Boosting - GBM
- sklearn.ensemble.GradientBoostingClassifier
- class sklearn.ensemble.GradientBoostingClassifier(*, loss='log_loss', learning_rate=0.1, n_estimators=100, subsample=1.0, criterion='friedman_mse', min_samples_split=2, min_samples_leaf=1, min_weight_fraction_leaf=0.0, max_depth=3, min_impurity_decrease=0.0, init=None, random_state=None, max_features=None, verbose=0, max_leaf_nodes=None, warm_start=False, validation_fraction=0.1, n_iter_no_change=None, tol=0.0001, ccp_alpha=0.0)

## ??? ????

In [ ]:
import warnings
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
warnings.filterwarnings('ignore')

# features.txt ???? ?? ?? index? ???? ???? ???? ??. ?? DataFrame?? ??.
feature_name_df = pd.read_csv('./datasets/UCI HAR Dataset/human_activity/features.txt',sep='\s+', header=None,
                             names=['column_index', 'column_name'])
# feature_name_df
# # ??? index? ????, ???? ??? ??? ??? ? ??? 10?? ??
feature_name = feature_name_df.iloc[:, 1].values.tolist()
print('?? ????? 10?? ??:', feature_name[:10])

feature_dup_df = feature_name_df.groupby('column_name').count()
print(feature_dup_df[feature_dup_df['column_index'] > 1].count())
feature_dup_df[feature_dup_df['column_index'] > 1].head()

In [ ]:
def get_new_feature_name_df(old_feature_name_df):
    feature_dup_df = pd.DataFrame(data=old_feature_name_df.groupby('column_name').cumcount(),
                                 columns=['dup_cnt'])
    feature_dup_df = feature_dup_df.reset_index()
    new_feature_name_df = pd.merge(old_feature_name_df.reset_index(), feature_dup_df, how='outer')
    new_feature_name_df['column_name'] = new_feature_name_df[['column_name', 'dup_cnt']].apply(lambda x : 
                                            x[0]+'-'+str(x[1])if x[1] >0 else x[0] , axis=1)
    
    new_feature_name_df = new_feature_name_df.drop(['index'], axis=1)
    return new_feature_name_df

In [ ]:
import pandas as pd
def get_human_dataset():
    # ? ??? ???? ???? ???? ???? read_csv?? ?? ??? sep?? ??.
    feature_name_df = pd.read_csv('./datasets/UCI HAR Dataset/human_activity/features.txt', sep='\s+',
                                 header=None,names=['column_index','column_name'])
    
    # ??? ???? ???? get_new_feature_name_df()? ??, ?? ??? DataFrame??.
    new_feature_name_df = get_new_feature_name_df(feature_name_df)
    
    #DataFrame? ???? ???? ???? ?? ??? ??? ?? ??
    feature_name = new_feature_name_df.iloc[:, 1].values.tolist()
    
    #?? ?? ??? ?? ??? ?? ???? DataFrame?? ??. ???? feature_name?? ??
    X_train = pd.read_csv('./datasets/UCI HAR Dataset/human_activity/train/X_train.txt', sep='\s+', names=feature_name)
    X_test = pd.read_csv('./datasets/UCI HAR Dataset/human_activity/test/X_test.txt', sep='\s+' , names=feature_name)
    
    # ?? ???? ??? ??? ???? DataFrame?? ???? ???? action?? ??.
    y_train = pd.read_csv('./datasets/UCI HAR Dataset/human_activity/train/y_train.txt' , sep='\s+', header=None, names=['action'])
    y_test = pd.read_csv('./datasets/UCI HAR Dataset/human_activity/test/y_test.txt', sep='\s+', header=None, names=['action'])
    
    # ??? ??/???? DataFrame? ?? ??
    return X_train, X_test, y_train, y_test

X_train, X_test, y_train, y_test = get_human_dataset()

## y? ???

In [ ]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()
encoder.fit(y_train)
y_train = encoder.transform(y_train)

encoder.fit(y_test)
y_test = encoder.transform(y_test)
y_test

## GBM ??

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
import time
import warnings
from sklearn.metrics import accuracy_score

warnings.filterwarnings('ignore')
X_train, X_test, y_train, y_test = get_human_dataset()
start_time = time.time()
gb_clf = GradientBoostingClassifier(random_state=0)
gb_clf.fit(X_train,y_train)
gb_pred = gb_clf.predict(X_test)
gb_accuracy = accuracy_score(y_test, gb_pred)
print(gb_accuracy)
print(time.time() - start_time)

# XGB ??

In [ ]:
from xgboost import XGBClassifier

# ?? ??
model = XGBClassifier(n_estimators=500, learning_rate=0.2, max_depth=4, random_state = 32)

# ?? ??
model.fit(X_train, y_train)

from sklearn.metrics import accuracy_score

y_pred = model.predict(X_test) # ?? ??(0? 1? ??)

# ?? ??? ?? ?? ??? ??? ??
accuracy_score(y_pred, y_test) 